## Analyzing Factors Associated with Traffic Crash Severity in California Using 2025 CCRS Data

**Team Members:**  
- Jonathan Mitchell  
- Daniel Ortega Rubio 
- Sherlyn Ballestero Cruz

**Research Question:**  
Are environmental factors (weather, lighting, road conditions) and temporal factors (time of day, day of the week) significantly associated with crash severity in California 2025 traffic crashes?

This notebook follows the full data science pipeline.


### Step 1: Get Data

Load the dataset, clean column names, select only useful columns and convert the date.


In [5]:
import pandas as pd
import numpy as np
import altair as alt
from altair import Chart, X, Y
from IPython.display import display

# Official dataset URL
url = "https://data.ca.gov/dataset/80c6a49d-c6b3-40ba-86d8-379c9741b4be/resource/9f4fc839-122d-4595-a146-43bc4ed16f46/download/crashes_2025.csv"

# Load the dataset
df = pd.read_csv(url, low_memory=False)

# Clean column names
df.columns = df.columns.str.strip()

# Keep only the columns we need (reduces memory a lot)
useful_cols = [
    'Collision Id', 'Crash Date Time', 'Day Of Week',
    'Weather 1', 'LightingDescription', 'Road Condition 1',
    'NumberInjured', 'NumberKilled', 'Collision Type Description'
]
df = df[useful_cols].copy()

print("Shape after selecting columns:", df.shape)
display(df.head())


Shape after selecting columns: (402539, 9)


,Collision Id,Crash Date Time,Day Of Week,Weather 1,LightingDescription,Road Condition 1,NumberInjured,NumberKilled,Collision Type Description
0,4550265,1/18/2025 3:20:00 PM,Saturday,CLEAR,DAYLIGHT,NO UNUSUAL CONDITIONS,1.0,0.0,OVERTURNED
1,4550264,1/10/2025 8:28:00 AM,Friday,CLEAR,DAYLIGHT,NO UNUSUAL CONDITIONS,1.0,0.0,REAR END
2,4550263,1/23/2025 8:45:00 AM,Thursday,CLEAR,DAYLIGHT,NO UNUSUAL CONDITIONS,0.0,0.0,SIDE SWIPE
3,4550262,1/15/2025 12:30:00 PM,Wednesday,WIND,DAYLIGHT,NO UNUSUAL CONDITIONS,0.0,0.0,HIT OBJECT
4,4550261,1/20/2025 12:30:00 PM,Monday,CLEAR,DAYLIGHT,NO UNUSUAL CONDITIONS,3.0,0.0,BROADSIDE


In [6]:
# Convert Crash Date Time to datetime
df['Crash Date Time'] = pd.to_datetime(
    df['Crash Date Time'],
    format='%m/%d/%Y %I:%M:%S %p',
    errors='coerce'
)


# Create Hour column (needed for analysis)
df['Hour'] = df['Crash Date Time'].dt.hour


print("Date range:", df['Crash Date Time'].min(), "to", df['Crash Date Time'].max())


Date range: 2025-01-01 00:00:00 to 2025-12-31 23:59:00


### Step 2: Exploratory Analysis and Visualization

We create a sample for visualization so Altair stays fast. All charts use pre-aggregated data.


In [7]:
# Create a sample for visualization (fast)
df_viz = df.sample(n=min(50000, len(df)), random_state=42).copy()

alt.data_transformers.disable_max_rows()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Chart 1 - Crashes by day of the week
day_counts = (
    df_viz.dropna(subset=['Day Of Week'])
          .groupby('Day Of Week')
          .size()
          .reset_index(name='count')
)

display(
    Chart(day_counts).mark_bar().encode(
        x=X('Day Of Week', sort=day_order, title='Day of Week'),
        y='count'
    ).properties(title='Number of Crashes by Day of Week', width=500)
)

# Chart 2 - Crashes by hour of the day
hour_counts = (
    df_viz.dropna(subset=['Hour'])
          .groupby('Hour')
          .size()
          .reset_index(name='count')
)

display(
    Chart(hour_counts).mark_bar().encode(
        x=X('Hour:O', title='Hour of Day'),
        y='count'
    ).properties(title='Number of Crashes by Hour of Day', width=600)
)

# Chart 3 - Top weather conditions
top_weather = df_viz['Weather 1'].value_counts().head(8).index
weather_counts = (
    df_viz[df_viz['Weather 1'].isin(top_weather)]
          .groupby('Weather 1')
          .size()
          .reset_index(name='count')
)

display(
    Chart(weather_counts).mark_bar().encode(
        x='count',
        y=Y('Weather 1', sort='-x', title='Weather')
    ).properties(title='Top Weather Conditions', width=500)
)

# Chart 4 - Lighting conditions
lighting_counts = (
    df_viz.dropna(subset=['LightingDescription'])
          .groupby('LightingDescription')
          .size()
          .reset_index(name='count')
)

display(
    Chart(lighting_counts).mark_bar().encode(
        x='count',
        y=Y('LightingDescription', sort='-x', title='Lighting')
    ).properties(title='Lighting Conditions', width=500)
)


alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

### Step 3: Data Cleaning

Create the Severity variable (vectorized = fast), clean categories, and drop rows with missing key values.


In [8]:
# Create Severity column (vectorized - much faster than apply)
df['Severity'] = np.where(
    df['NumberKilled'] > 0, 'Fatal',
    np.where(df['NumberInjured'] > 0, 'Injury', 'PDO')
)

print("Severity distribution:")
print(df['Severity'].value_counts(normalize=True).round(3))

# Clean Weather (keep top categories, rest = Other)
top_weather = df['Weather 1'].value_counts().nlargest(7).index
df['Weather_Clean'] = df['Weather 1'].where(df['Weather 1'].isin(top_weather), 'Other')

# Clean Lighting
top_lighting = df['LightingDescription'].value_counts().nlargest(5).index
df['Lighting_Clean'] = df['LightingDescription'].where(
    df['LightingDescription'].isin(top_lighting), 'Other'
)

# Drop rows missing critical variables
df_clean = df.dropna(subset=['Severity', 'Weather_Clean', 'Lighting_Clean', 'Hour', 'Day Of Week']).copy()

print("\nShape after cleaning:", df_clean.shape)
print("\nWeather_Clean categories:")
print(df_clean['Weather_Clean'].value_counts())


Severity distribution:
Severity
PDO       0.581
Injury    0.411
Fatal     0.008
Name: proportion, dtype: float64

Shape after cleaning: (402539, 13)

Weather_Clean categories:
Weather_Clean
CLEAR             343607
CLOUDY             40040
RAINING            14310
FOG/VISIBILITY      1762
Other               1636
WIND                 444
SNOWING              424
OTHER                316
Name: count, dtype: int64


### Step 4: Rescaling / Encoding

Most features are categorical. Use One-Hot Encoding before modeling.


In [ ]:
# TODO: Separate features (X) and target (y)

# TODO: Encode categorical variables

# TODO: (Optional) Scale numeric features if the model requires it


### Step 5: Training / Test Split


In [ ]:

# TODO: Split the data into training and test sets (use stratification)

# TODO: Check the class distribution in the training set and in the test set

### Step 6: Model Building

Example: Random Forest (handles class imbalance with class_weight).


In [ ]:
# TODO: Perform statistical test(s) to evaluate association between factors and Severity

# TODO: Build and train a model using the training data

# TODO: (Optional) Try a second approach and compare results


### Step 7: Model Testing


In [ ]:
# TODO: Generate predictions on the test set

# TODO: Calculate and print performance metrics

# TODO: Create and display the confusion matrix

# TODO: Examine which features contribute most to the predictions (if applicable)

### Step 8: Polishing and Presenting

### Conclusions (complete after running the analysis)

**Main findings:**
- TODO

**Answer to the research question:**
- TODO

**Limitations:**
- TODO

**Recommendations:**
- TODO
